<a href="https://colab.research.google.com/github/Ikbal-ullah/JE-Early-Warning-System/blob/master/Phase5_modification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 1. Restore the Enterprise Geospatial Environment
!pip install -q rasterio geopandas rasterstats shapely h3

import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import h3
from rasterstats import zonal_stats
import rasterio
from rasterio.windows import from_bounds
import os
import numpy as np
from google.colab import files

print("--- INITIATING CORRECTED PHASE 5: BASE-RISK + AMPLIFIER ENGINE ---")

pig_raster_path = "glw4_pigs.tif"

if not os.path.exists(pig_raster_path):
    raise FileNotFoundError(f"CRITICAL HALT: {pig_raster_path} not found. Ensure it is uploaded to Colab.")

print("Loading Masked Phase 4 Matrix...")
df = pd.read_csv("nalbari_phase4_masked_hazard.csv.gz")

unique_hexagons = df['hexagon'].unique()
print(f"Generating spatial polygons for {len(unique_hexagons)} vector zones...")

hex_polygons = [{"hexagon": hex_id, "geometry": Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(hex_id)])} for hex_id in unique_hexagons]
gdf_hex = gpd.GeoDataFrame(hex_polygons, crs="EPSG:4326")

with rasterio.open(pig_raster_path) as src:
    raster_crs = src.crs
    print(f"Reprojecting vector zones to align with UN spatial coordinates ({raster_crs})...")
    gdf_hex_proj = gdf_hex.to_crs(raster_crs)

    minx, miny, maxx, maxy = gdf_hex_proj.total_bounds

    print("Cropping GLW4 Livestock raster to Nalbari bounds...")
    window = from_bounds(minx, miny, maxx, maxy, src.transform)
    transform = src.window_transform(window)
    cropped_pigs = src.read(1, window=window)
    nodata_value = src.nodata

print("Executing Zonal Summation for Swine Density...")
pig_stats = zonal_stats(gdf_hex_proj, cropped_pigs, affine=transform, stats="mean", nodata=nodata_value, all_touched=True)
gdf_hex['pig_population'] = [stat['mean'] * 0.73 if stat['mean'] is not None else 0 for stat in pig_stats]

print("Applying the Corrected Eco-Triad Math (The Real-World Fix)...")
df_final = pd.merge(df, gdf_hex[['hexagon', 'pig_population']], on='hexagon', how='left')

# THE MATHEMATICAL FIX
# Adding 1.0 to ensure zero-pig zones retain their base climate/hydrology hazard
df_final['pig_multiplier'] = 1.0 + np.log1p(df_final['pig_population'])
df_final['triad_spillover_hazard'] = df_final['final_spillover_hazard'] * df_final['pig_multiplier']

print("\n--- RECALIBRATING RISK THRESHOLDS (Unsupervised Classification) ---")
active_hazards = df_final[df_final['triad_spillover_hazard'] > 0]['triad_spillover_hazard']

threshold_warning = np.percentile(active_hazards, 75)
threshold_critical = np.percentile(active_hazards, 90)

print(f"  -> Level 2 (Warning) Threshold: {threshold_warning:,.2f}")
print(f"  -> Level 3 (Critical) Threshold: {threshold_critical:,.2f}")

def classify_risk(hazard):
    if hazard == 0:
        return 0  # Safe
    elif hazard < threshold_warning:
        return 1  # Monitor
    elif hazard < threshold_critical:
        return 2  # Warning
    else:
        return 3  # Critical Action

print("Classifying spatial-temporal records...")
df_final['alert_level'] = df_final['triad_spillover_hazard'].apply(classify_risk)

output_file = "nalbari_je_radar_final_classified.csv.gz"
print(f"Compressing Final Radar Output...")
df_final.to_csv(output_file, index=False, compression="gzip")

print(f"Pipeline Complete. Triggering download for {output_file}...")
files.download(output_file)

print("\n--- SYSTEM SUMMARY (Total Days Across All Zones) ---")
print(df_final['alert_level'].value_counts().sort_index().rename({
    0: 'Level 0 (Safe)',
    1: 'Level 1 (Monitor)',
    2: 'Level 2 (Warning)',
    3: 'Level 3 (Critical)'
}))

--- INITIATING CORRECTED PHASE 5: BASE-RISK + AMPLIFIER ENGINE ---
Loading Masked Phase 4 Matrix...
Generating spatial polygons for 2671 vector zones...
Reprojecting vector zones to align with UN spatial coordinates (EPSG:4326)...
Cropping GLW4 Livestock raster to Nalbari bounds...
Executing Zonal Summation for Swine Density...
Applying the Corrected Eco-Triad Math (The Real-World Fix)...

--- RECALIBRATING RISK THRESHOLDS (Unsupervised Classification) ---
  -> Level 2 (Warning) Threshold: 408,808.12
  -> Level 3 (Critical) Threshold: 671,101.75
Classifying spatial-temporal records...
Compressing Final Radar Output...
Pipeline Complete. Triggering download for nalbari_je_radar_final_classified.csv.gz...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- SYSTEM SUMMARY (Total Days Across All Zones) ---
alert_level
Level 0 (Safe)         511166
Level 1 (Monitor)     1810184
Level 2 (Warning)      362037
Level 3 (Critical)     241358
Name: count, dtype: int64


In [6]:
!pip install -q folium h3 pandas

import pandas as pd
import h3
import folium
import json
from google.colab import files

print("--- GENERATING CORRECTED GEOSPATIAL RADAR MAP ---")

print("Loading Real-World Classified Data...")
df = pd.read_csv("nalbari_je_radar_final_classified.csv.gz")

print("Aggregating Chronic Risk Hotspots...")
level_3_data = df[df['alert_level'] == 3]
hotspots = level_3_data.groupby('hexagon').size().reset_index(name='level_3_days')

all_hexes = pd.DataFrame({'hexagon': df['hexagon'].unique()})
hotspots = pd.merge(all_hexes, hotspots, on='hexagon', how='left').fillna(0)

print("Building Interactive Folium Map Engine...")
m = folium.Map(location=[26.44, 91.44], zoom_start=11, tiles="CartoDB positron")

features = []
for _, row in hotspots.iterrows():
    hex_id = row['hexagon']
    risk_days = row['level_3_days']

    boundary = h3.cell_to_boundary(hex_id)
    geom = [[ [lng, lat] for lat, lng in boundary ]]

    if risk_days > 150:
        color = "#ff0000" # Severe Red
        fill_opacity = 0.6
    elif risk_days > 75:
        color = "#ff6600" # Orange
        fill_opacity = 0.5
    elif risk_days > 0:
        color = "#ffcc00" # Yellow
        fill_opacity = 0.4
    else:
        color = "#00ff00" # Green (Mathematically Safe)
        fill_opacity = 0.1

    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": geom
        },
        "properties": {
            "hexagon_id": hex_id,
            "days_at_level_3": risk_days,
            "style": {
                "fillColor": color,
                "color": color,
                "weight": 1,
                "fillOpacity": fill_opacity
            }
        }
    }
    features.append(feature)

geojson_data = {"type": "FeatureCollection", "features": features}

folium.GeoJson(
    geojson_data,
    style_function=lambda x: x['properties']['style'],
    tooltip=folium.GeoJsonTooltip(fields=['hexagon_id', 'days_at_level_3'], aliases=['Vector Zone:', 'Total Days at Level 3:'])
).add_to(m)

map_file = "Nalbari_JEV_Corrected_Map.html"
print("Rendering map interface...")
m.save(map_file)

print(f"Map successfully generated. Triggering download for {map_file}...")
files.download(map_file)

--- GENERATING CORRECTED GEOSPATIAL RADAR MAP ---
Loading Real-World Classified Data...
Aggregating Chronic Risk Hotspots...
Building Interactive Folium Map Engine...
Rendering map interface...
Map successfully generated. Triggering download for Nalbari_JEV_Corrected_Map.html...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>